In [1]:
from nltk import word_tokenize
from nltk.corpus import stopwords
from pymongo import MongoClient
import pandas as pd

from Youtube.youtube_api_scripts.config import MONGO_DB, MONGO_URI, MONGO_COLL
import matplotlib.pyplot as plt

In [2]:

with MongoClient(MONGO_URI) as client:
    db = client[MONGO_DB]
    collection = db[MONGO_COLL]

    limit_results = 0
    # Fetch the latest 5000 documents sorted by updated_at in descending order
    with collection.find().sort('updated_at', -1).limit(limit_results) as cursor:
        data = pd.DataFrame(list(cursor))

data

,_id,comment_id,author,author_channel_id,channel_id,channel_name,like_count,published_at,text,updated_at,video_id,video_title
0,6848fa490cbff59905a1b042,UgzhKup3CAtVDDm-Lh94AaABAg,@siddrekt5136,{'value': 'UCe6Wf0Wrcg-4C4ZnK_F4Sog'},UCyf0SCaSGhKpvI_azfvidUw,MarcelianOnline,0,2025-06-11T03:33:33Z,Going into dungeons with my favorite spec will...,2025-06-11T03:33:33Z,LnTmU_Jk6tM,Is WoW Still ACTUALLY Fun to Play Today?
1,6848fa490cbff59905a1b043,UgwlzfWjzoQaJG29GOl4AaABAg,@skitteronline,{'value': 'UCsEbtZuy8Tj5izazBjzQ0ww'},UCyf0SCaSGhKpvI_azfvidUw,MarcelianOnline,0,2025-06-11T03:29:45Z,was playing from vanilla ant left after legion...,2025-06-11T03:31:05Z,LnTmU_Jk6tM,Is WoW Still ACTUALLY Fun to Play Today?
2,6848fa490cbff59905a1b044,Ugzijr7_vFZSewEM8ux4AaABAg,@heman147r,{'value': 'UCRT6mJjvLLJBsBSHUVo4JVw'},UCyf0SCaSGhKpvI_azfvidUw,MarcelianOnline,0,2025-06-11T03:26:39Z,Yall want a good warrior class?....go play ffx...,2025-06-11T03:26:39Z,TBLrtVZznZk,The Warrior Rework WE NEED (But Might Never Get…)
3,6848fa490cbff59905a1b045,UgwC63uLbCMf5BVglpp4AaABAg,@kinbailey310,{'value': 'UCqZ3EdMbFk3UoXLpSFUobXA'},UCyf0SCaSGhKpvI_azfvidUw,MarcelianOnline,0,2025-06-11T02:59:45Z,I wish we had epic raids like we use to. Like ...,2025-06-11T02:59:45Z,LnTmU_Jk6tM,Is WoW Still ACTUALLY Fun to Play Today?
4,6848fd540cbff59905a1b952,UgyNeR-qJYFB4_LKyC94AaABAg,@muscadineman45pritchett82,{'value': 'UCX1L1bNooTI8LmbVcSXN7vA'},UCL1w_mREZgr1NXpRz_i088Q,Venruki,0,2025-06-11T02:54:55Z,Im so happy. I can finally play retail again,2025-06-11T02:54:55Z,j4i6K2tBXQw,Blizzard Are Killing Addons In PvP... And I Th...
...,...,...,...,...,...,...,...,...,...,...,...,...
3279559,6788ed51eaf88b06c403ede3,UgxM9w7zIATkRtamuBl4AaABAg,@JontePlugg,{'value': 'UCKSRJ5ZgnH5kAPryZN0LWmw'},UCvV80raFP2R7X0yxH9iJjog,GingiTV,0,2013-09-16T15:22:37Z,did you config elvui yourself? or is that a co...,1970-01-01T00:00:00Z,K4Hn34aWues,Scarlet Monastery 4:30
3279560,6788ed51eaf88b06c403ede4,Ugx_GeswWh8hfeof7pV4AaABAg,@GingiTV,{'value': 'UCvV80raFP2R7X0yxH9iJjog'},UCvV80raFP2R7X0yxH9iJjog,GingiTV,0,2013-09-15T17:39:04Z,"Salyin Battle Banner, drops from a dude in val...",1970-01-01T00:00:00Z,K4Hn34aWues,Scarlet Monastery 4:30
3279561,6788ed51eaf88b06c403ede5,UgxA_glbUquVlJNFTsp4AaABAg,@Luiskaah,{'value': 'UCqL1SNyTen4xV6c5pSdVEiA'},UCvV80raFP2R7X0yxH9iJjog,GingiTV,0,2013-09-14T19:45:10Z,Hello! Really nice run. What do you use to res...,1970-01-01T00:00:00Z,K4Hn34aWues,Scarlet Monastery 4:30
3279562,6788ed51eaf88b06c403ede6,Ugw4kFnZtwOOoXsJxdR4AaABAg,@ZortherPwnz,{'value': 'UCUxPIFICN0ugugp9vECumaA'},UCvV80raFP2R7X0yxH9iJjog,GingiTV,0,2013-09-14T13:29:36Z,So me the way master !\n,1970-01-01T00:00:00Z,4gu7VPFrXK0,Scarlet Halls 3:50


In [ ]:
data.info()

In [4]:
import re

from bs4 import BeautifulSoup
import ftfy

def clean_text(text):
    # Fix unicode issues
    text = ftfy.fix_text(text)
    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    # Lowercase
    text = text.lower()
    # Remove email addresses
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
     # Remove non-alphanumeric characters except '+' (keep spaces and numbers)
    text = re.sub(r'[^a-z0-9\s\+]', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Demonstration
sample = '<p>My key is +15 M+. hmu me@example.com!</p>'
print(clean_text(sample))

my key is +15 m+ hmu


In [19]:
df = data.copy()
df['updated_at'] = pd.to_datetime(df['updated_at'])
df.loc[df['updated_at']=='1970-01-01 00:00:00+00:00', 'updated_at'] = df['published_at']

In [20]:
df = df[
    ['channel_name', 'like_count', 'published_at', 'updated_at','video_title',
     'text']]

In [21]:
print(df['video_title'].isna().count())

3279564


In [22]:
df[df['video_title'].isna()]

,channel_name,like_count,published_at,updated_at,video_title,text
7691,MrGM,0,2025-05-30T01:58:32Z,2025-05-30 01:58:32+00:00,None,Wasnt Dark Portal-themed effect for the Hearth...
367037,Dvalin Gaming,1,2024-06-28T20:14:54Z,2024-06-28 20:14:54+00:00,None,cheers was wandering how i get higher level co...
379605,Dvalin Gaming,1,2024-06-14T20:34:09Z,2024-06-14 20:34:09+00:00,None,Butt you lost up and down FOV. What if the ene...
381749,Dvalin Gaming,2,2024-06-12T09:12:32Z,2024-06-12 09:12:32+00:00,None,"All your alts will be in the same warband, the..."
381770,Dvalin Gaming,1,2024-06-12T08:32:10Z,2024-06-12 08:32:10+00:00,None,❤❤
...,...,...,...,...,...,...
1145807,MoreMrGM,0,2022-04-07T13:21:05Z,2022-04-07 13:21:05+00:00,None,"It has some issues, yes, but i really like thi..."
1145867,MoreMrGM,3,2022-04-07T10:38:20Z,2022-04-07 10:38:20+00:00,None,"I legit hope Blizz keep this in, I understand ..."
1145882,MoreMrGM,31,2022-04-07T10:08:54Z,2022-04-07 10:08:54+00:00,None,Great information and knowledge as always. It'...
1149844,MoreMrGM,0,2022-04-03T13:38:13Z,2022-04-03 13:38:13+00:00,None,How is BoD gonna work😂


In [25]:
# Fill NaNs in the column "video_title" with the value "Shorts"
df['video_title']=df['video_title'].fillna('Shorts')